In [1]:
from IPython.display import Javascript, display

test_js = """
navigator.mediaDevices.enumerateDevices().then(devices => {
  const mics = devices.filter(d => d.kind === 'audioinput');
  if (mics.length === 0) {
    console.log("NO MICROPHONE DEVICES FOUND");
  } else {
    console.log("Microphones found:", mics.length);
    mics.forEach(m => console.log(" -", m.label || "(unnamed device)"));
  }
  navigator.mediaDevices.getUserMedia({audio: true})
    .then(() => console.log("Mic permission: GRANTED"))
    .catch(e => console.log("Mic permission: DENIED -", e.message));
});
"""
display(Javascript(test_js))
print("Check the browser console (F12 -> Console tab) for results.")

<IPython.core.display.Javascript object>

Check the browser console (F12 -> Console tab) for results.


In [2]:
# ================================================================
# 🤖 BIG AI CHATBOT PROJECT — Colab Single Cell
# ================================================================
# Features:
#  1. Custom intent classifier (trained from scratch, PyTorch)
#  2. Open-domain chat fallback using DialoGPT (chat about ANYTHING)
#  3. Sentiment analysis on user messages (VADER)
#  4. Voice input (mic) + voice output (gTTS)
#  5. Save / load conversation history (JSON)
#  6. Gradio web UI with chat, voice, sentiment, history download
# ================================================================

# --- 0. Install dependencies ---
!pip install -q torch transformers nltk gTTS SpeechRecognition pydub gradio vaderSentiment PyPDF2
!apt-get -y install ffmpeg -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import re
import io
import json
import os
from datetime import datetime
from base64 import b64decode

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from gtts import gTTS
from IPython.display import Audio, display, Javascript
import speech_recognition as sr
from pydub import AudioSegment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sentiment_analyzer = SentimentIntensityAnalyzer()

import PyPDF2

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# ================================================================
# PART 1 — CUSTOM INTENT MODEL (trained from scratch)
# ================================================================

intents = {
  "intents": [
    {"tag": "greeting",
     "patterns": ["hi", "hello", "hey", "good morning", "good evening", "what's up", "howdy"],
     "responses": ["Hello! How can I help you today?", "Hi there!", "Hey! Nice to see you."]},

    {"tag": "goodbye",
     "patterns": ["bye", "goodbye", "see you later", "talk to you later", "quit", "exit"],
     "responses": ["Goodbye! Have a great day.", "See you soon!", "Bye! Take care."]},

    {"tag": "thanks",
     "patterns": ["thanks", "thank you", "that's helpful", "appreciate it"],
     "responses": ["You're welcome!", "Happy to help!", "Anytime!"]},

    {"tag": "name",
     "patterns": ["what is your name", "who are you", "what should I call you"],
     "responses": ["I'm your AI lab chatbot, built from scratch with a transformer brain attached!", "You can call me ChatBot."]},

    {"tag": "how_are_you",
     "patterns": ["how are you", "how are you doing", "how do you feel"],
     "responses": ["I'm just a program, but I'm running great!", "All systems operational, thanks for asking!"]},

    {"tag": "help",
     "patterns": ["help", "what can you do", "how can you help me", "capabilities"],
     "responses": ["I can chat about almost anything, recognize your mood, and talk back with voice!",
                    "Ask me anything — small talk, deep questions, or just say hi."]},

    {"tag": "weather",
     "patterns": ["what's the weather", "is it raining", "weather today", "how's the weather"],
     "responses": ["I can't check live weather, but I hope it's nice outside!", "I don't have weather access, sorry!"]},

    {"tag": "joke",
     "patterns": ["tell me a joke", "make me laugh", "say something funny"],
     "responses": ["Why don't scientists trust atoms? Because they make up everything!",
                    "I told my computer I needed a break, and it said no problem — it'll go to sleep."]},

    {"tag": "ai",
     "patterns": ["what is ai", "what is artificial intelligence", "explain machine learning", "what is deep learning"],
     "responses": ["AI is the simulation of human intelligence by machines.",
                    "Machine learning lets computers learn patterns from data without explicit rules."]},

    {"tag": "creator",
     "patterns": ["who made you", "who created you", "who built you", "who is your developer"],
     "responses": ["I was built as part of an AI lab project!", "A student trained me from scratch for an AI project."]},

    {"tag": "math",
     "patterns": ["solve math", "calculate", "what is 5 plus 5", "help with algebra", "math problem"],
     "responses": ["I can help solve math problems.", "Let's work through the calculation together."]},

    {"tag": "programming",
     "patterns": ["python", "java", "coding help", "programming", "write code"],
     "responses": ["I can help with programming questions.", "Tell me what code you need help with."]},

    {"tag": "science",
     "patterns": ["what is science", "physics", "chemistry", "biology", "scientific explanation"],
     "responses": ["Science helps us understand the natural world.", "I can explain scientific concepts."]},

    {"tag": "history",
     "patterns": ["history", "historical event", "world war", "ancient civilization"],
     "responses": ["History helps us understand past events.", "Ask me about historical topics."]},

    {"tag": "sports",
     "patterns": ["football", "cricket", "basketball", "sports news", "sports"],
     "responses": ["Sports are a great way to stay active.", "I can discuss many sports topics."]},

    {"tag": "movies",
     "patterns": ["movie recommendation", "best movie", "film", "cinema"],
     "responses": ["I enjoy discussing movies.", "Tell me your favorite genre."]},

    {"tag": "music",
     "patterns": ["music", "song", "singer", "band", "playlist"],
     "responses": ["Music is a universal language.", "What kind of music do you like?"]},

    {"tag": "technology",
     "patterns": ["technology", "latest tech", "computer", "smartphone", "innovation"],
     "responses": ["Technology evolves very quickly.", "I can discuss technology topics."]},

    {"tag": "health",
     "patterns": ["health tips", "healthy lifestyle", "exercise", "fitness"],
     "responses": ["A balanced lifestyle supports good health.", "Exercise and proper sleep are important."]},

    {"tag": "education",
     "patterns": ["study tips", "education", "learning", "university", "school"],
     "responses": ["Learning is a lifelong process.", "I can provide study suggestions."]},

    {"tag": "cybersecurity",
     "patterns": ["cybersecurity", "hacking", "computer security", "online safety"],
     "responses": ["Cybersecurity helps protect digital systems.", "Strong passwords improve security."]},

    {"tag": "robotics",
     "patterns": ["robot", "robotics", "automation", "autonomous system"],
     "responses": ["Robotics combines engineering and AI.", "Robots are used in many industries."]},

    {"tag": "machine_learning",
     "patterns": ["machine learning", "ml", "supervised learning", "unsupervised learning"],
     "responses": ["Machine Learning enables computers to learn from data.", "ML is a major branch of AI."]},

    {"tag": "deep_learning",
     "patterns": ["deep learning", "neural network", "cnn", "rnn"],
     "responses": ["Deep Learning uses multi-layer neural networks.", "Deep learning powers many modern AI systems."]},

    {"tag": "database",
     "patterns": ["database", "sql", "mysql", "sqlite", "data storage"],
     "responses": ["Databases are used to store and manage information.", "SQL is commonly used to query databases."]},

    {"tag": "networking",
     "patterns": ["network", "ip address", "router", "internet connection"],
     "responses": ["Computer networks connect devices together.", "Networking is essential for communication."]},

    {"tag": "github",
     "patterns": ["github", "repository", "git commit", "git push"],
     "responses": ["GitHub is widely used for software development.", "I can help with Git and GitHub."]},

    {"tag": "linux",
     "patterns": ["ubuntu", "linux command", "terminal", "bash"],
     "responses": ["Linux is a powerful operating system.", "I can help with Linux commands."]},

    {"tag": "project",
     "patterns": ["project idea", "final year project", "ai project", "software project"],
     "responses": ["I can suggest project ideas.", "Tell me your project requirements."]},

    {"tag": "motivation",
     "patterns": ["motivate me", "inspiration", "encouragement", "feeling lazy"],
     "responses": ["Small progress every day leads to big results.", "Keep learning and improving."]},

    {"tag": "time",
     "patterns": ["what time is it", "current time", "tell me the time"],
     "responses": ["I don't always know the current time, but your device does."]},

    {"tag": "date",
     "patterns": ["today's date", "what date is today", "current date"],
     "responses": ["You can check your device for the exact current date."]},

    {"tag": "language",
     "patterns": ["translate", "language", "english to bangla", "bangla to english"],
     "responses": ["I can help with language translation.", "Tell me the text you want translated."]},

    {"tag": "career",
     "patterns": ["career advice", "job", "future career", "interview"],
     "responses": ["Building skills consistently helps career growth.", "I can provide career guidance."]}
  ]
}

def tokenize(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z0-9'\s]", "", sentence)
    tokens = nltk.word_tokenize(sentence)
    return [lemmatizer.lemmatize(t) for t in tokens]

all_words, tags, xy = [], [], []
for intent in intents['intents']:
    tag = intent['tag']
    tags.append(tag)
    for pattern in intent['patterns']:
        w = tokenize(pattern)
        all_words.extend(w)
        xy.append((w, tag))

all_words = sorted(set(all_words))
tags = sorted(set(tags))

def bag_of_words(tokenized_sentence, words):
    bag = np.zeros(len(words), dtype=np.float32)
    for idx, w in enumerate(words):
        if w in tokenized_sentence:
            bag[idx] = 1.0
    return bag

X_train, y_train = [], []
for (pattern_sentence, tag) in xy:
    X_train.append(bag_of_words(pattern_sentence, all_words))
    y_train.append(tags.index(tag))

X_train = np.array(X_train)
y_train = np.array(y_train)
print(f"[Intent Model] Vocabulary: {len(all_words)} | Tags: {tags} | Samples: {len(X_train)}")

class ChatDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()
    def __getitem__(self, index):
        return self.X[index], self.y[index]
    def __len__(self):
        return len(self.X)

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        out = self.relu(self.l1(x))
        out = self.dropout(out)
        out = self.relu(self.l2(out))
        out = self.dropout(out)
        return self.l3(out)

input_size = len(all_words)
hidden_size = 16
output_size = len(tags)
batch_size = 8
learning_rate = 0.001
num_epochs = 300

dataset = ChatDataset(X_train, y_train)
train_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

intent_model = NeuralNet(input_size, hidden_size, output_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(intent_model.parameters(), lr=learning_rate)

print("[Intent Model] Training...")
for epoch in range(num_epochs):
    for words_batch, labels in train_loader:
        words_batch, labels = words_batch.to(device), labels.to(device)
        outputs = intent_model(words_batch)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f'  Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

intent_model.eval()
print("[Intent Model] Training complete!\n")

CONFIDENCE_THRESHOLD = 0.75  # high threshold: only catch CLEAR matches, else fall to transformer

def get_intent_response(sentence):
    tokens = tokenize(sentence)
    bag = bag_of_words(tokens, all_words)
    bag = torch.from_numpy(bag).to(device).unsqueeze(0)

    with torch.no_grad():
        output = intent_model(bag)
        probs = torch.softmax(output, dim=1)
        prob, predicted = torch.max(probs, dim=1)

    tag = tags[predicted.item()]
    confidence = prob.item()

    if confidence > CONFIDENCE_THRESHOLD:
        for intent in intents['intents']:
            if intent['tag'] == tag:
                return random.choice(intent['responses']), tag, confidence

    return None, "unknown", confidence

# ================================================================
# PART 2 — OPEN-DOMAIN CHAT (DialoGPT, "chat about anything")
# ================================================================

print("[DialoGPT] Loading pretrained conversational model...")
DIALOGPT_MODEL = "microsoft/DialoGPT-medium"
dialo_tokenizer = AutoTokenizer.from_pretrained(DIALOGPT_MODEL)
dialo_model = AutoModelForCausalLM.from_pretrained(DIALOGPT_MODEL).to(device)
print("[DialoGPT] Ready!\n")

# Per-session chat history (token ids) for context
chat_history_ids = None

def get_dialogpt_response(user_text):
    global chat_history_ids
    new_input_ids = dialo_tokenizer.encode(user_text + dialo_tokenizer.eos_token, return_tensors='pt').to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Keep context window manageable
    if bot_input_ids.shape[-1] > 800:
        bot_input_ids = bot_input_ids[:, -800:]

    chat_history_ids = dialo_model.generate(
        bot_input_ids,
        max_length=bot_input_ids.shape[-1] + 60,
        pad_token_id=dialo_tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.75,
    )

    reply = dialo_tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    return reply.strip() if reply.strip() else "Hmm, tell me more about that."

def reset_dialogpt_history():
    global chat_history_ids
    chat_history_ids = None

# ================================================================
# PART 2b — SUMMARIZATION (BART, summarize any passage/paragraph)
# ================================================================

print("[Summarizer] Loading pretrained summarization model...")
SUMMARIZER_MODEL = "facebook/bart-large-cnn"
summarizer_tokenizer = AutoTokenizer.from_pretrained(SUMMARIZER_MODEL)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZER_MODEL).to(device)
summarizer_model.eval()
print("[Summarizer] Ready!\n")

def _run_summarizer(text, max_len, min_len):
    inputs = summarizer_tokenizer(
        text, return_tensors="pt", max_length=1024, truncation=True
    ).to(device)
    with torch.no_grad():
        summary_ids = summarizer_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            min_length=min_len,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def summarize_text(text, summary_length="medium"):
    text = text.strip()
    if not text:
        return "Please provide some text to summarize."

    word_count = len(text.split())
    if word_count < 15:
        return "Text is too short to summarize — please provide a longer passage."

    # Map desired length to max/min token lengths, scaled to input size
    length_settings = {
        "short":  (0.25, 15),
        "medium": (0.45, 25),
        "long":   (0.65, 40),
    }
    ratio, min_len = length_settings.get(summary_length, length_settings["medium"])
    max_len = max(min_len + 10, int(word_count * ratio))
    max_len = min(max_len, 250)  # BART limit safeguard

    # BART has a 1024-token input limit — chunk long text and summarize each chunk
    max_chunk_words = 700
    words = text.split()

    if len(words) <= max_chunk_words:
        chunks = [text]
    else:
        chunks = [" ".join(words[i:i + max_chunk_words]) for i in range(0, len(words), max_chunk_words)]

    summaries = []
    for chunk in chunks:
        chunk_word_count = len(chunk.split())
        chunk_max_len = min(max_len, max(min_len + 10, int(chunk_word_count * ratio)))
        chunk_min_len = min(min_len, chunk_max_len - 5) if chunk_max_len > min_len else 5
        try:
            summaries.append(_run_summarizer(chunk, chunk_max_len, chunk_min_len))
        except Exception as e:
            summaries.append(f"[Error summarizing chunk: {e}]")

    combined = " ".join(summaries)

    # If multiple chunks were summarized, do a final pass to tighten it up
    if len(chunks) > 1 and len(combined.split()) > max_len:
        try:
            combined = _run_summarizer(combined, max_len, min_len)
        except Exception:
            pass

    return combined

# ================================================================
# PART 2c — PDF READER (extract text from PDF + summarize)
# ================================================================

def extract_pdf_text(pdf_path, max_pages=None):
    """Extract text from a PDF file. Returns (full_text, num_pages, error)."""
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            num_pages = len(reader.pages)
            pages_to_read = num_pages if max_pages is None else min(max_pages, num_pages)

            text_parts = []
            for i in range(pages_to_read):
                page = reader.pages[i]
                page_text = page.extract_text() or ""
                text_parts.append(page_text)

            full_text = "\n".join(text_parts).strip()
            return full_text, num_pages, None
    except Exception as e:
        return "", 0, str(e)

def summarize_pdf(pdf_path, summary_length="medium", max_pages=None):
    """Extract text from a PDF and summarize it."""
    full_text, num_pages, error = extract_pdf_text(pdf_path, max_pages=max_pages)

    if error:
        return "", f"Error reading PDF: {error}", 0

    if not full_text.strip():
        return "", "No extractable text found (this PDF may be a scanned image without OCR).", num_pages

    summary = summarize_text(full_text, summary_length)
    return full_text, summary, num_pages


# ================================================================
# PART 3 — SENTIMENT ANALYSIS
# ================================================================

def analyze_sentiment(text):
    scores = sentiment_analyzer.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.5:
        label = "Positive 😊"
    elif compound <= -0.5:
        label = "Negative 😞"
    else:
        label = "Neutral 😐"
    return label, compound

# ================================================================
# PART 4 — MASTER RESPONSE FUNCTION (intent first, transformer fallback)
# ================================================================

conversation_log = []  # list of dicts: {time, user, bot, tag, sentiment, score}

SUMMARIZE_TRIGGERS = [
    r"^summarize[:\s]+(.+)",
    r"^summarise[:\s]+(.+)",
    r"^summary[:\s]+(.+)",
    r"^tl;?dr[:\s]+(.+)",
]

def detect_summarize_request(text):
    for pattern in SUMMARIZE_TRIGGERS:
        match = re.match(pattern, text.strip(), re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()
    return None

def get_bot_response(user_text):
    sentiment_label, sentiment_score = analyze_sentiment(user_text)

    # Check if the user is asking to summarize something
    passage = detect_summarize_request(user_text)
    if passage:
        reply = "📄 **Summary:**\n" + summarize_text(passage, "medium")
        tag, source = "summarize", "summarizer"
        conversation_log.append({
            "time": datetime.now().strftime("%H:%M:%S"),
            "user": user_text,
            "bot": reply,
            "tag": tag,
            "source": source,
            "sentiment": sentiment_label,
            "sentiment_score": round(sentiment_score, 3)
        })
        return reply, tag, source, sentiment_label, sentiment_score

    intent_reply, tag, conf = get_intent_response(user_text)

    if intent_reply is not None:
        reply = intent_reply
        source = "intent"
    else:
        reply = get_dialogpt_response(user_text)
        source = "dialogpt"
        tag = "open_domain"

    conversation_log.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "user": user_text,
        "bot": reply,
        "tag": tag,
        "source": source,
        "sentiment": sentiment_label,
        "sentiment_score": round(sentiment_score, 3)
    })

    return reply, tag, source, sentiment_label, sentiment_score

# ================================================================
# PART 5 — VOICE I/O
# ================================================================

def speak(text, filename="response.mp3"):
    try:
        tts = gTTS(text=text, lang='en')
        tts.save(filename)
        return filename
    except Exception as e:
        print(f"TTS error: {e}")
        return None

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async () => {
    blob = new Blob(chunks);
    text = await new Promise(resolve => {
      reader = new FileReader();
      reader.onloadend = () => resolve(reader.result.split(',')[1]);
      reader.readAsDataURL(blob);
    });
    resolve(text);
  };
  recorder.stop();
});
"""

def record_audio(seconds=4, filename="input.wav"):
    from google.colab import output as colab_output
    print(f"Recording for {seconds} seconds... speak now!")
    display(Javascript(RECORD_JS))
    b64 = colab_output.eval_js(f'record({seconds * 1000})')
    audio_bytes = b64decode(b64)
    audio = AudioSegment.from_file(io.BytesIO(audio_bytes))
    audio.export(filename, format="wav")
    return filename

def transcribe_audio(filename="input.wav"):
    recognizer = sr.Recognizer()
    with sr.AudioFile(filename) as source:
        audio_data = recognizer.record(source)
    try:
        return recognizer.recognize_google(audio_data)
    except sr.UnknownValueError:
        return None
    except sr.RequestError as e:
        return f"[Error: {e}]"

# ================================================================
# PART 6 — SAVE / LOAD CONVERSATION HISTORY
# ================================================================

HISTORY_FILE = "chat_history.json"

def save_history(filename=HISTORY_FILE):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(conversation_log, f, indent=2, ensure_ascii=False)
    return filename

def load_history(filename=HISTORY_FILE):
    global conversation_log
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            conversation_log = json.load(f)
        return f"Loaded {len(conversation_log)} past messages."
    return "No history file found."

# ================================================================
# PART 7 — GRADIO WEB UI
# ================================================================

def gradio_chat(message, history, show_debug):
    reply, tag, source, sentiment_label, sentiment_score = get_bot_response(message)
    if show_debug:
        debug = f"\n\n_(source: {source} | intent: {tag} | sentiment: {sentiment_label} {sentiment_score:+.2f})_"
        return reply + debug
    return reply

def gradio_voice_input(audio_filepath):
    if audio_filepath is None:
        return "No audio received.", None, ""
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_filepath) as source:
            audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data)
    except Exception:
        return "Could not understand audio.", None, ""

    reply, tag, source, sentiment_label, sentiment_score = get_bot_response(text)
    audio_out = speak(reply, "ui_response.mp3")
    info = f"You said: {text}\nIntent: {tag} | Source: {source} | Sentiment: {sentiment_label} ({sentiment_score:+.2f})"
    return reply, audio_out, info

def gradio_save_history():
    fname = save_history()
    return fname

def gradio_clear():
    global conversation_log
    conversation_log = []
    reset_dialogpt_history()
    return None

def gradio_summarize(text, length_choice):
    if not text or not text.strip():
        return "Please paste some text to summarize."
    length_map = {"Short": "short", "Medium": "medium", "Long": "long"}
    return summarize_text(text, length_map.get(length_choice, "medium"))

def gradio_pdf_reader(pdf_file, length_choice, max_pages_input):
    if pdf_file is None:
        return "", "Please upload a PDF file.", ""

    length_map = {"Short": "short", "Medium": "medium", "Long": "long"}
    summary_length = length_map.get(length_choice, "medium")

    max_pages = None
    if max_pages_input and str(max_pages_input).strip():
        try:
            max_pages = int(max_pages_input)
            if max_pages <= 0:
                max_pages = None
        except ValueError:
            max_pages = None

    full_text, summary, num_pages = extract_pdf_text(pdf_file, max_pages=max_pages)

    info = f"Total pages in PDF: {num_pages}"
    if max_pages:
        info += f" (read first {min(max_pages, num_pages)} pages)"

    # Truncate displayed extracted text if very long, to keep UI responsive
    display_text = full_text
    if len(display_text) > 20000:
        display_text = display_text[:20000] + "\n\n...[truncated for display]..."

    return display_text, summary, info

with gr.Blocks(title="AI Lab Chatbot") as demo:
    gr.Markdown("# 🤖 AI Lab Chatbot\n"
                "Custom-trained intent model + DialoGPT for open-domain chat, "
                "with sentiment analysis, voice support, text summarization, and PDF reading.")

    with gr.Tab("💬 Text Chat"):
        debug_toggle = gr.Checkbox(label="Show debug info (source / intent / sentiment)", value=False)
        chatbot_ui = gr.ChatInterface(
            fn=gradio_chat,
            chatbot=gr.Chatbot(height=400),
            textbox=gr.Textbox(placeholder="Say anything — greetings, jokes, or deep questions..."),
            additional_inputs=[debug_toggle]
        )

    with gr.Tab("🎤 Voice Chat"):
        gr.Markdown("Record a message, the bot will reply in text **and** voice.")
        audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Speak here")
        voice_btn = gr.Button("Send Voice Message")
        voice_reply = gr.Textbox(label="Bot reply (text)")
        voice_audio_out = gr.Audio(label="Bot reply (voice)", autoplay=True)
        voice_info = gr.Textbox(label="Details (transcript, intent, sentiment)")
        voice_btn.click(gradio_voice_input, inputs=audio_input,
                         outputs=[voice_reply, voice_audio_out, voice_info])

    with gr.Tab("📝 Summarize"):
        gr.Markdown("Paste any passage, article, or paragraph below to get a concise summary.\n\n"
                     "💡 You can also type `summarize: <your text>` directly in the Text Chat tab.")
        summarize_input = gr.Textbox(label="Text to summarize", lines=10,
                                      placeholder="Paste a long passage, article, or paragraph here...")
        summarize_length = gr.Radio(["Short", "Medium", "Long"], value="Medium", label="Summary length")
        summarize_btn = gr.Button("Summarize")
        summarize_output = gr.Textbox(label="Summary", lines=6)
        summarize_btn.click(gradio_summarize, inputs=[summarize_input, summarize_length], outputs=summarize_output)

    with gr.Tab("📄 PDF Reader"):
        gr.Markdown("Upload a PDF to extract its text and get an AI-generated summary.\n\n"
                     "💡 For large PDFs, optionally limit the number of pages read (e.g. for a quick preview).")
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
        with gr.Row():
            pdf_length = gr.Radio(["Short", "Medium", "Long"], value="Medium", label="Summary length")
            pdf_max_pages = gr.Textbox(label="Max pages to read (optional, blank = all)", placeholder="e.g. 5")
        pdf_btn = gr.Button("Read & Summarize PDF")
        pdf_info = gr.Textbox(label="Info", interactive=False)
        pdf_summary = gr.Textbox(label="Summary", lines=6)
        pdf_extracted = gr.Textbox(label="Extracted Text (preview)", lines=12)
        pdf_btn.click(gradio_pdf_reader, inputs=[pdf_input, pdf_length, pdf_max_pages],
                       outputs=[pdf_extracted, pdf_summary, pdf_info])

    with gr.Tab("📊 History & Tools"):
        gr.Markdown("Save the current conversation log to a JSON file, or clear it.")
        save_btn = gr.Button("Save Conversation History")
        save_out = gr.File(label="Download history file")
        clear_btn = gr.Button("Clear History / Reset Context")
        clear_status = gr.Textbox(label="Status", interactive=False)

        save_btn.click(gradio_save_history, outputs=save_out)
        clear_btn.click(lambda: (gradio_clear(), "History cleared, context reset.")[1],
                         outputs=clear_status)

print("\nLaunching Gradio app...\n")
demo.launch(share=True, debug=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Using device: cpu
[Intent Model] Vocabulary: 182 | Tags: ['ai', 'career', 'creator', 'cybersecurity', 'database', 'date', 'deep_learning', 'education', 'github', 'goodbye', 'greeting', 'health', 'help', 'history', 'how_are_you', 'joke', 'language', 'linux', 'machine_learning', 'math', 'motivation', 'movies', 'music', 'name', 'networking', 'programming', 'project', 'robotics', 'science', 'sports', 'technology', 'thanks', 'time', 'weather'] | Samples: 144
[Intent Model] Training...
  Epoch [100/300], Loss: 0.9661
  Epoch [200/300], Loss: 1.2854
  Epoch [300/300], Loss: 0.8144
[Intent Model] Training complete!

[DialoGPT] Loading pretrained conversational model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

[Summarizer] Ready!



/tmp/ipykernel_1847/3552956008.py:702: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=400),
/tmp/ipykernel_1847/3552956008.py:702: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(height=400),
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(



Launching Gradio app...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a71a5ee11ef97d80c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
